# UTTA-Med — remaining optional GPU (seed 42)

Temperature scaling (hospital 1) · SAR · EATA-C · DLTTA · Camelyon17-C

**Frozen source.** No retrain. `bn_freeze_stats`, lr=`1e-5`. Do not retune if a method looks worse than source.

In [ ]:
import os, sys, glob, shutil
from pathlib import Path
os.chdir('/kaggle/working')
for z in glob.glob('/kaggle/working/*.zip') + glob.glob('/kaggle/input/**/*.zip', recursive=True):
    if 'optional' in z.lower() or 'UTTA-Med-remaining' in z:
        print('unzip', z)
        shutil.unpack_archive(z, '/kaggle/working')
        break
PROJ = None
for c in [Path('/kaggle/working'), Path('/kaggle/working/UTTA-Med')]:
    if (c/'src'/'tta'/'sar.py').exists():
        PROJ = c; break
if PROJ is None:
    hits = list(Path('/kaggle/working').rglob('src/tta/sar.py'))
    assert hits, 'unzip UTTA-Med-remaining-optional.zip into /kaggle/working'
    PROJ = hits[0].parents[2]
sys.path.insert(0, str(PROJ))
os.chdir(PROJ)
print('PROJ', PROJ)
!pip -q install pyarrow scikit-learn tqdm pyyaml huggingface_hub datasets pillow
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
import runpy
from pathlib import Path
script = Path('notebooks/kaggle_remaining_optional.py')
if not script.exists():
    script = Path('kaggle_remaining_optional.py')
print('running', script.resolve())
runpy.run_path(str(script), run_name='__main__')

In [ ]:
import json
from pathlib import Path
p = Path('/kaggle/working/camelyon17_remaining_optional_s42.json')
d = json.loads(p.read_text())
print('T =', round(d['temperature']['T'], 4))
print('ECE raw/scaled', round(d['temperature']['raw']['ece'],4), round(d['temperature']['scaled']['ece'],4))
print('source AUROC', round(d['source_test']['auroc'], 4))
for k in ('sar_test','eata_c_test','dltta_test'):
    a = d[k]['after']
    print(f"{k:16s} AUROC={a['auroc']:.4f} F1={a['f1']:.4f} harm={d[k]['harm']['harm_rate']:.4f} cov={d[k]['coverage']:.3f}")
print('Camelyon17-C rows', len(d.get('camelyon17_c_source', [])))
print('Download', p)